<a href="https://colab.research.google.com/github/Lattemelia/Portforio/blob/main/demand%20forecasting1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 新しいセクション

In [34]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [35]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [36]:
PATH = r"/content/drive/MyDrive/Colab Notebooks"

test_df = pd.read_csv(PATH + r"/test.csv")
train_df = pd.read_csv(PATH + r"/train.csv")

marged_df = pd.concat([test_df,train_df],axis=0)

In [53]:
marged_df.isnull().sum()

,0
id,0
brand,0
model,0
model_year,0
milage,0
fuel_type,8466
engine,0
transmission,0
ext_col,0
int_col,0


In [50]:
#エンジン名のみ抽出
root_regex = r"(?:^\d+\.\d+HP\s+)?([^\s]+)(?:\s+(.+))?$"

# 元の engine 列から直接、一撃で2カラムに代入
marged_df[["engine_displacement", "engine_type"]] = marged_df["engine"].str.extract(root_regex)


In [104]:
import re
import numpy as np
import pandas as pd


def literal_absolute_perfect_engine_classifier(row):
    # 文字列の標準化
    engine_str = (
        str(row["engine"]).lower().strip() if pd.notna(row["engine"]) else ""
    )
    model_str = (
        str(row["model"]).lower().strip() if pd.notna(row["model"]) else ""
    )

    model_clean = (
        model_str.replace(" ", "")
        .replace("-", "")
        .replace("/", "")
        .replace(".", "")
    )
    combined_text = model_clean + " " + engine_str

    # ─── 【ステップ1】100%純電気自動車（BEV）だけの「絶対隔離」 ───
    # ※ハイブリッド(HV)やプラグインハイブリッド(PHEV)はエンジンを持つため、ここでは除外します。
    pure_bev_models = [
        "modely",
        "model3",
        "modelx",
        "models",
        "r1s",
        "r1t",
        "airgrand",
        "airpure",
        "mustangmache",
        "f150lightning",
        "etron",
        "ev6",
        "bolteuv",
        "boltev",
        "leaf",
        "lyriq",
        "konaev",
        "id4",
        "bz4x",
        "egolf",
        "c40recharge",
        "500e",
        "eqs450",
    ]

    is_pure_bev = False
    # 明確に純EVモデル名が含まれている場合
    if any(m in model_clean for m in pure_bev_models):
        is_pure_bev = True
    # engine列が「電気モーターのみ」を示していて、かつハイブリッドやガソリンの表記がない場合
    elif "electric motor" in engine_str and not any(
        x in combined_text for x in ["hybrid", "phev", "plugin", "gas"]
    ):
        is_pure_bev = True
    elif (engine_str == "electric" or engine_str == "ev") and not any(
        x in model_clean for x in ["solara", "hybrid", "phev"]
    ):
        is_pure_bev = True

    # プラグインハイブリッドやレンジエクステンダー（i3, XC90, Revero等）は純EVから除外して次へ進める
    if any(x in combined_text for x in ["phev", "plugin", "hybrid", "recharge", "grandtouring", "i3", "revero"]):
        is_pure_bev = False

    # 純EVであれば、ここで「Other、0.0L」で確定終了
    if is_pure_bev:
        return pd.Series(["Other", 0.0])

    # ─── 【ステップ2】車種名からの排気量(L)・気筒数の事前マッピング（超強力救済） ───
    displacement = np.nan
    engine_type = "Unknown"

    # モデル名に気筒数の決定的なヒントがある場合、最優先で型と排気量をセット
    if "solara" in model_clean:
        engine_type, displacement = "V6", 3.3
    elif "ftype" in model_clean:
        engine_type, displacement = "V6", 3.0
    elif "macan" in model_clean:
        engine_type, displacement = "V6", 3.0
    elif "450h" in model_clean:
        engine_type, displacement = "V6", 3.5
    elif "55" in model_clean and any(x in model_clean for x in ["a8", "a6", "q8", "a7", "q7"]):
        engine_type, displacement = "V6", 3.0
    elif "40i" in model_clean or "440i" in model_clean or "340i" in model_clean or "m40i" in model_clean:
        engine_type, displacement = "V6", 3.0
    elif "53" in model_clean and "amg" in model_clean:
        engine_type, displacement = "V6", 3.0
    elif "63" in model_clean and "amg" in model_clean:
        engine_type, displacement = "V8", 4.0
    elif "580" in model_clean or "560" in model_clean or "550" in model_clean:
        engine_type, displacement = "V8", 4.0
    elif "392" in model_clean or "srt" in model_clean:
        engine_type, displacement = "V8", 6.2
    elif "huracan" in model_clean or "r8" in model_clean:
        engine_type, displacement = "V10", 5.2
    elif "aventador" in model_clean:
        engine_type, displacement = "V12/W16", 6.5
    elif "flyingspur" in model_clean:
        engine_type, displacement = "V8", 4.0
    elif "i3" in model_clean:
        engine_type, displacement = "I3", 0.65
    elif "revero" in model_clean:
        engine_type, displacement = "I4", 2.0
    elif any(x in model_clean for x in ["prius", "ct200h", "rav4", "evoque", "q3", "x3xdrive30i", "ilx", "kicks"]):
        engine_type, displacement = "I4", 2.0

    # ─── 【ステップ3】テキストからの排気量（リッター数）の自動抽出（未確定の場合のみ） ───
    if pd.isna(displacement):
        match_l = re.search(r"(\d+\.\d+|\d+)\s*(l|t|liter|litres)", engine_str)
        if match_l:
            try: displacement = float(match_l.group(1))
            except ValueError: pass

    if pd.isna(displacement) or displacement > 10.0:
        match_num = re.search(r"\b(\d+\.\d+)\b", engine_str + " " + model_str)
        if match_num:
            try:
                val = float(match_num.group(1))
                if 0.5 <= val <= 8.5: displacement = val
            except ValueError: pass

    # ─── 【ステップ4】気筒数（エンジンタイプ）の分類（未確定の場合のみ） ───
    if engine_type == "Unknown":
        if any(x in combined_text for x in ["12cyl", "v12", "w16", "16cyl"]):
            engine_type = "V12/W16"
        elif any(x in combined_text for x in ["10cyl", "v10"]):
            engine_type = "V10"
        elif any(x in combined_text for x in ["8cyl", "v8"]):
            engine_type = "V8"
        elif any(x in combined_text for x in ["6cyl", "v6", "i6", "inline6", "flat6", "h6", "mhev"]):
            engine_type = "V6"
        elif any(x in combined_text for x in ["5cyl", "i5"]):
            engine_type = "I5"
        elif any(x in combined_text for x in ["4cyl", "i4", "inline4", "hybrid", "phev", "plugin"]):
            engine_type = "I4"
        elif any(x in combined_text for x in ["3cyl", "i3"]):
            engine_type = "I3"
        elif "rotary" in combined_text or "rx8" in model_clean:
            engine_type = "Rotary"

    # ─── 【ステップ5】★最終防衛ライン：気筒数が不明な場合、排気量から逆算。情報ゼロなら適正救済 ───
    if engine_type == "Unknown":
        if pd.notna(displacement) and displacement > 0:
            if displacement >= 6.0: engine_type = "V12/W16"
            elif displacement >= 4.0: engine_type = "V8"
            elif displacement >= 2.6: engine_type = "V6"
            elif displacement >= 1.3: engine_type = "I4"
            else: engine_type = "I3"
        else:
            # 本当に何一つ情報がないガソリン車・HVの救済
            if any(t in model_clean for t in ["f150", "1500", "sierra", "silverado", "tundra", "ram", "expedition", "tahoe", "armada", "wagoneer"]):
                engine_type, displacement = "V6", 3.5
            else:
                engine_type, displacement = "I4", 2.0

    return pd.Series([engine_type, displacement])


# データフレームへの一括適用
marged_df[["engine_type_clean", "displacement_l"]] = marged_df.apply(
    literal_absolute_perfect_engine_classifier, axis=1
)

# 📊 結果の最終確認
df8 = marged_df[marged_df["engine_type_clean"] == "Other"]
print(df8["model"].value_counts())

model
Model Y Long Range                                1323
Model Y Performance                                788
R1S Adventure Package                              779
Model 3 Long Range                                 677
Camry Solara SLE V6                                652
Model X Long Range Plus                            562
Model X 75D                                        415
Model X Long Range                                 359
Model S 100D                                       339
Model S P100D                                      318
Mustang Mach-E GT                                  281
Model 3 Standard Range Plus                        211
R1S Launch Edition                                 200
Model X Base                                       170
F-150 Lightning XLT                                169
Taycan Base                                        162
Model 3 Performance                                141
Taycan Turbo                                       140
Mode

In [105]:
import re
import numpy as np
import pandas as pd


def extract_displacement(row):
    # すでにEV（Electric）または水素車（FCEV）と判定されているものは排気量 0.0
    if row["engine_type_clean"] in ["Electric", "FCEV (Hydrogen)"]:
        return 0.0

    engine_str = (
        str(row["engine"]).lower().strip() if pd.notna(row["engine"]) else ""
    )

    # 1. 「数字.数字l」または「数字l」（例: 3.5l, 2l, 2.0t の 2.0）を探す正規表現
    # 💡 2.0t や 3.0l のような表記を幅広くキャッチします
    match_l = re.search(r"(\d+\.\d+|\d+)\s*(l|t|liter)", engine_str)
    if match_l:
        try:
            return float(match_l.group(1))
        except ValueError:
            pass

    # 2. 救済措置：l や liter が書いておらず「3.5 V6」のように数字と気筒数だけ並んでいる場合
    match_num = re.search(r"\b(\d+\.\d+)\b", engine_str)
    if match_num:
        try:
            val = float(match_num.group(1))
            # 💡 排気量として現実的な数値（0.6 から 8.5リッターの間）であれば採用
            if 0.6 <= val <= 8.5:
                return val
        except ValueError:
            pass

    # どうしても見つからない場合は欠損値（または一目でわかるように np.nan や 0.0）にする
    return np.nan


# 💡 新しい排気量カラム「displacement_l」を生成
marged_df["displacement_l"] = marged_df.apply(extract_displacement, axis=1)

# ─── 📊 抽出結果の確認 ───

print("🚗 【確認1】EV（Electric）の排気量がすべて0になっているか:")
print(
    marged_df[marged_df["engine_type_clean"] == "Electric"][
        "displacement_l"
    ].value_counts()
)

print("\n🔥 【確認2】ガソリン車・HV（V6やV8など）の主な排気量まとめ（上位20件）:")
# 欠損値をのぞくガソリン車の排気量分布
print(marged_df["displacement_l"].dropna().value_counts().head(20))

print("\n📋 【確認3】エンジンタイプ別の「平均排気量」一覧:")
print(marged_df.groupby("engine_type_clean")["displacement_l"].mean())

🚗 【確認1】EV（Electric）の排気量がすべて0になっているか:
Series([], Name: count, dtype: int64)

🔥 【確認2】ガソリン車・HV（V6やV8など）の主な排気量まとめ（上位20件）:
displacement_l
3.0    43233
2.0    38637
3.5    36800
4.0    18319
6.2    17975
3.6    16863
2.5    11586
5.3    10654
5.7    10320
5.0    10133
3.8     8810
4.4     7251
4.6     6903
3.7     5809
2.4     5667
4.7     4889
2.7     4511
6.7     4162
6.0     3715
5.5     3048
Name: count, dtype: int64

📋 【確認3】エンジンタイプ別の「平均排気量」一覧:
engine_type_clean
I3         1.276817
I4         2.152256
Other      3.295375
Rotary     1.442342
V10        4.556457
V12/W16    6.308157
V6         3.361941
V8         4.941931
Name: displacement_l, dtype: float64


In [108]:
df8 = marged_df[marged_df["engine_type_clean"] == "Other"]
df8["model"].value_counts()

,count
model,
Model Y Long Range,1323
Model Y Performance,788
R1S Adventure Package,779
Model 3 Long Range,677
Camry Solara SLE V6,652
Model X Long Range Plus,562
Model X 75D,415
Model X Long Range,359
Model S 100D,339


In [107]:
# 1. 判定が「Other」または「Unknown_Engine」になってしまっているデータを抽出
leaked_data = marged_df[
    marged_df["engine_type_clean"].isin(["Other", "Unknown_Engine"])
]

# 2. その中から、純EV（Teslaなど）を除外して、「本当はガソリン/HVなのに漏れている車」を特定
#    （※もし排気量が 0.0L 以外の Other があれば、それが判定漏れのガソリン車です）
actual_leaks = leaked_data[leaked_data["displacement_l"] > 0.0]

print(f"判別できなかったガソリン車/HVの件数: {len(actual_leaks)}件")
print("\n=== 漏れているデータの具体的な中身（先頭30件） ===")
print(actual_leaks[["model", "engine"]].head(30))

判別できなかったガソリン車/HVの件数: 854件

=== 漏れているデータの具体的な中身（先頭30件） ===
                            model  \
156           Camry Solara SLE V6   
250           Camry Solara SLE V6   
327           Camry Solara SLE V6   
388                   F-TYPE V6 S   
771            Model Y Long Range   
1186          Camry Solara SLE V6   
1911          Camry Solara SLE V6   
1978          Camry Solara SLE V6   
2060          Camry Solara SLE V6   
2212          Camry Solara SLE V6   
2321          Camry Solara SLE V6   
2483          Camry Solara SLE V6   
2792          Camry Solara SLE V6   
3112          Camry Solara SLE V6   
3297          Camry Solara SLE V6   
3466          Camry Solara SLE V6   
3801        R1S Adventure Package   
4704                    e-Golf SE   
5557  Q4 e-tron Sportback Premium   
5598          Camry Solara SLE V6   
6609                  EV6 GT-Line   
6705          Camry Solara SLE V6   
6718          Camry Solara SLE V6   
7202          Camry Solara SLE V6   
7477          Cam